In [0]:
import mlflow
import pandas as pd

EXPERIMENT_NAME = "/Users/2519868@esan.edu.pe/control-2"
MODEL_NAME = "control_2_model"
ALIAS_NAME = "PRINCIPAL"
METRIC_NAME = "roc_auc"

mlflow.set_experiment(EXPERIMENT_NAME)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Experiment ID: {experiment.experiment_id}")

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=[f"metrics.{METRIC_NAME} DESC"],
    max_results=50,
    output_format="pandas"
)

if runs_df.empty:
    raise ValueError(f"No se encontraron runs en el experimento {EXPERIMENT_NAME}.")

runs_df = runs_df.sort_values(by=f"metrics.{METRIC_NAME}", ascending=False, na_position="last").reset_index(drop=True)

best_run = runs_df.iloc[0]
best_run_id = best_run["run_id"]
best_metric_value = best_run[f"metrics.{METRIC_NAME}"]

print("=== Mejor run encontrado ===")
print(best_run[["run_id", "tags.mlflow.runName", f"metrics.{METRIC_NAME}"]])
print(f"Run seleccionado: {best_run_id}")
print(f"Valor de {METRIC_NAME}: {best_metric_value}")

In [0]:
from mlflow.exceptions import RestException

model_uri = f"runs:/{best_run_id}/model"

print(f"Registrando el modelo desde: {model_uri}")
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=MODEL_NAME,
)

print("=== Modelo registrado ===")
print(f"Name: {registered_model.name}")
print(f"Version: {registered_model.version}")

try:
    client.delete_registered_model_alias(name=MODEL_NAME, alias=ALIAS_NAME)
    print(f"Alias {ALIAS_NAME} ya existía; se eliminó para actualizarlo.")
except RestException:
    print(f"Alias {ALIAS_NAME} no existía aún.")

client.set_registered_model_alias(
    name=MODEL_NAME,
    alias=ALIAS_NAME,
    version=registered_model.version,
)

print(f"Alias {ALIAS_NAME} asignado a la versión {registered_model.version} del modelo {MODEL_NAME}.")

In [0]:
print("=== Resumen final ===")
print(f"Modelo principal: {MODEL_NAME}")
print(f"Alias asignado: {ALIAS_NAME}")
print(f"Versión registrada: {registered_model.version}")
print(f"Run base: {best_run_id}")
print(f"Métrica usada: {METRIC_NAME} = {best_metric_value}")